# Cross-Validating the AST Taxonomy with Semantic Embeddings (Q5, Top-K)

Independent check on the AST-based structural taxonomy from the companion
paper: does a code-embedding model, given **no knowledge of which AST rule
matched a submission**, recover the same manual/builtin/novel split that the
structural competencies (`SelectionStep`, `FindExtremum`) already assign?

**Design:**
1. Label every Q5 submission by its structural match: `manual`
   (`SelectionStep`), `builtin` (`FindExtremum` but not `SelectionStep`), or
   `novel` (matches neither, i.e. `FindExtremum_final == 0`).
2. Split 70% train / 30% test.
3. Build one embedding centroid per label from the training 70%.
4. Classify the held-out 30% by nearest centroid, using embeddings only.
5. Compare the semantic prediction against the real structural label.

**What this tells you either way:**
- High agreement -> real, independent evidence the AST taxonomy's categories
  correspond to genuine distinctions in embedding space, not an artifact of
  the rules themselves.
- Low agreement, especially on `novel` -> a legitimate finding worth
  reporting: either the embedding model isn't separating these cases well,
  or `novel` isn't one coherent strategy and needs sub-clustering rather than
  a single centroid (`novel` is "everything the rules didn't recognize", not
  one implementation approach -- worth checking directly if that row looks
  weak, see the note at the end of this notebook).

Requires: your Drive folder already has `competency.py`, `topk_report.csv`,
and the Refactory Q5 `correct/`/`wrong/` `.py` files (same layout as the
semantic-backup-validation notebook).


## 1. Setup

In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (fine for codet5p-110m-embedding on short snippets)")

import transformers
print("transformers version:", transformers.__version__)


## 2. Mount Google Drive

Same `PROJECT_DIR` convention as the semantic-backup-validation notebook --
adjust the path if your folder is named differently.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/competency_extraction'  # <-- adjust if needed

import os
assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR} -- check the folder name/path in Drive"
print("Contents:", os.listdir(PROJECT_DIR))


## 3. Load the embedding model

`Salesforce/codet5p-110m-embedding` -- trained with a contrastive objective
for code similarity (unlike raw `codebert-base`, which collapses under mean
pooling; see the semantic-backup-validation notebook for that finding).

Includes the config patch needed to work around a compatibility bug in the
model's 2023-era remote code (`AttributeError:
'CodeT5pEmbeddingConfig' object has no attribute 'is_decoder'` on recent
`transformers` -- this patch avoids needing any version pin).


In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
import torch

_checkpoint = "Salesforce/codet5p-110m-embedding"
_device = "cuda" if torch.cuda.is_available() else "cpu"

_tokenizer = AutoTokenizer.from_pretrained(_checkpoint, trust_remote_code=True)

_config = AutoConfig.from_pretrained(_checkpoint, trust_remote_code=True)
for _attr, _default in [("is_decoder", False), ("is_encoder_decoder", True),
                         ("pad_token_id", 0), ("eos_token_id", 2)]:
    if not hasattr(_config, _attr):
        setattr(_config, _attr, _default)

_model = AutoModel.from_pretrained(_checkpoint, config=_config, trust_remote_code=True).to(_device)
_model.eval()

@torch.no_grad()
def codebert_embed(code: str):
    inputs = _tokenizer.encode(code, return_tensors="pt", truncation=True, max_length=512).to(_device)
    embedding = _model(inputs)[0]  # already L2-normalized: norm == 1.0
    return embedding.cpu().numpy()

def cosine_sim(a, b) -> float:
    import numpy as np
    return float(np.dot(a, b))

# Sanity check -- should show a real gap, not near-identical scores
related_a = "def f(lst):\n    best = lst[0]\n    for x in lst:\n        if x > best:\n            best = x\n    return best"
related_b = "def f(lst):\n    return max(lst)"
unrelated  = "def greet(name):\n    return 'Hello, ' + name"
print("related vs related:", cosine_sim(codebert_embed(related_a), codebert_embed(related_b)))
print("related vs unrelated:", cosine_sim(codebert_embed(related_a), codebert_embed(unrelated)))


## 4. Load the batch report and assign structural labels

Reads `topk_report.csv` (from `batch_report.py`'s Q5 run) and labels every
submission `manual`, `builtin`, `novel`, or `partial` based on which
structural competency actually matched.


In [ ]:
import pandas as pd

report = pd.read_csv(f"{PROJECT_DIR}/topk_report.csv")

def structural_label(row, threshold=0.99):
    if row["SelectionStep_final"] >= threshold:
        return "manual"
    elif row["FindExtremum_final"] >= threshold:
        return "builtin"
    elif row["FindExtremum_final"] == 0:
        return "novel"
    else:
        return "partial"  # nonzero but matches neither branch fully -- excluded below

report["structural_label"] = report.apply(structural_label, axis=1)
print(report["structural_label"].value_counts())


## 5. 70/30 split

`SHUFFLE_BEFORE_SPLIT` defaults to `False` -- a literal first-70%/last-30%
split on `topk_report.csv`'s existing row order. Note that
`batch_report.py` builds this CSV via `sorted(glob.glob(...))`, so rows are
in alphabetical filename order -- if `correct_5_*.py` and `wrong_5_*.py`
happen to cluster unevenly under that ordering, the split may be less random
than you'd want. Flip the flag to `True` for a shuffled split instead (fixed
seed, so it's reproducible) if you'd rather rule that out.


In [ ]:
SHUFFLE_BEFORE_SPLIT = True  # flip to True for a randomized, reproducible split instead
SPLIT_SEED = 0

# Keep only the three clean groups this experiment is about
labeled = report[report["structural_label"].isin(["manual", "builtin", "novel"])].reset_index(drop=True)

if SHUFFLE_BEFORE_SPLIT:
    labeled = labeled.sample(frac=1, random_state=SPLIT_SEED).reset_index(drop=True)

split_point = int(len(labeled) * 0.7)
train_df = labeled.iloc[:split_point]
test_df = labeled.iloc[split_point:]

print(f"train: {len(train_df)}  test: {len(test_df)}")
print("\ntrain label counts:\n", train_df["structural_label"].value_counts())
print("\ntest label counts:\n", test_df["structural_label"].value_counts())


## 6. Build one centroid per label from the training 70%


In [ ]:
import os
import numpy as np

FOLDER = f"{PROJECT_DIR}/data/question_5/code"

def read_src(fname):
    for sub in ["correct", "wrong"]:
        p = f"{FOLDER}/{sub}/{fname}"
        if os.path.exists(p):
            return open(p, encoding="utf-8").read()
    return None

centroids = {}
for label in ["manual", "builtin", "novel"]:
    files = train_df[train_df["structural_label"] == label]["file"].tolist()
    embeddings = [codebert_embed(read_src(f)) for f in files if read_src(f) is not None]
    centroids[label] = np.mean(embeddings, axis=0)
    print(f"{label}: {len(embeddings)} training submissions embedded")


## 7. Classify the held-out 30% by nearest centroid

Embeddings only -- the structural label is used solely to score the
prediction afterward, never as an input to the classification itself.


In [ ]:
rows = []
for _, row in test_df.iterrows():
    src = read_src(row["file"])
    if src is None:
        continue
    emb = codebert_embed(src)
    sims = {label: cosine_sim(emb, c) for label, c in centroids.items()}
    predicted = max(sims, key=sims.get)
    rows.append({
        "file": row["file"],
        "structural_label": row["structural_label"],
        "semantic_predicted": predicted,
        **{f"sim_{k}": v for k, v in sims.items()},
    })

results_df = pd.DataFrame(rows)
results_df.head(10)


## 8. Confusion matrix and agreement


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

labels_order = ["manual", "builtin", "novel"]
cm = confusion_matrix(results_df["structural_label"], results_df["semantic_predicted"], labels=labels_order)

print("confusion matrix [rows=structural label, cols=semantic prediction]")
print(pd.DataFrame(cm, index=labels_order, columns=labels_order))

print("\noverall accuracy:", accuracy_score(results_df["structural_label"], results_df["semantic_predicted"]))
print("\n", classification_report(results_df["structural_label"], results_df["semantic_predicted"], labels=labels_order))


## 9. If the `novel` row looks weak: check whether `novel` is even one cluster

`novel` means "the rules didn't recognize this" -- it isn't guaranteed to be
one coherent implementation strategy. This cell shows how tightly (or
loosely) the real `novel` submissions actually cluster around their own
centroid, and which specific files are the closest/furthest from it -- useful
for deciding whether `novel` needs to be split into sub-strategies rather
than treated as one group.


In [ ]:
novel_test = results_df[results_df["structural_label"] == "novel"].copy()
novel_test = novel_test.sort_values("sim_novel", ascending=False)

print(f"novel test submissions: {len(novel_test)}")
print(f"sim_novel stats: mean={novel_test['sim_novel'].mean():.3f} std={novel_test['sim_novel'].std():.3f}")
print("\nclosest to the novel centroid (most 'typical' novel submissions):")
print(novel_test[["file", "sim_novel", "sim_manual", "sim_builtin"]].head(5))
print("\nfurthest from the novel centroid (most atypical -- possibly a distinct sub-strategy):")
print(novel_test[["file", "sim_novel", "sim_manual", "sim_builtin"]].tail(5))


## 10. Save results


In [ ]:
results_df.to_csv(f"{PROJECT_DIR}/ast_taxonomy_crossvalidation_results.csv", index=False)

from google.colab import files
files.download(f"{PROJECT_DIR}/ast_taxonomy_crossvalidation_results.csv")


## 11. Multi-seed robustness check
   
   Runs the same classification across several random train/test splits
   instead of one, to check how stable the manual/builtin agreement figure
   actually is -- this is what goes into the paper instead of a single run.
   

In [ ]:
# Pre-embed every file once. Embeddings don't depend on the train/test
# split, so there's no need to recompute them per seed.
embedding_cache = {}
for fname in labeled["file"]:
    src = read_src(fname)
    if src is not None:
        embedding_cache[fname] = codebert_embed(src)

print(f"cached {len(embedding_cache)} embeddings out of {len(labeled)} files")


In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

SEEDS = list(range(10))  # 10 runs; bump this up if you want tighter estimates
labels_order = ["manual", "builtin", "novel"]

per_seed_results = []
all_predictions = []  # pooled across every seed, for aggregated precision/recall

for seed in SEEDS:
    shuffled = labeled.sample(frac=1, random_state=seed).reset_index(drop=True)
    split_point = int(len(shuffled) * 0.7)
    train_df_s = shuffled.iloc[:split_point]
    test_df_s = shuffled.iloc[split_point:]

    # Build centroids from cached embeddings -- no re-embedding
    centroids_s = {}
    for label in labels_order:
        files = train_df_s[train_df_s["structural_label"] == label]["file"].tolist()
        vecs = [embedding_cache[f] for f in files if f in embedding_cache]
        centroids_s[label] = np.mean(vecs, axis=0)

    # Classify test set by nearest centroid
    rows = []
    for _, row in test_df_s.iterrows():
        f = row["file"]
        if f not in embedding_cache:
            continue
        emb = embedding_cache[f]
        sims = {label: cosine_sim(emb, c) for label, c in centroids_s.items()}
        predicted = max(sims, key=sims.get)
        rows.append({"file": f, "structural_label": row["structural_label"], "semantic_predicted": predicted})

    results_s = pd.DataFrame(rows)
    results_s["seed"] = seed
    all_predictions.append(results_s)

    # Full 3-way accuracy
    acc_3way = accuracy_score(results_s["structural_label"], results_s["semantic_predicted"])

    # Manual/builtin-only accuracy -- the figure actually reported in the paper
    mb_only = results_s[results_s["structural_label"].isin(["manual", "builtin"])]
    acc_mb = accuracy_score(mb_only["structural_label"], mb_only["semantic_predicted"])

    # Novel recall specifically, since that's the unstable one
    novel_rows = results_s[results_s["structural_label"] == "novel"]
    novel_recall = (novel_rows["semantic_predicted"] == "novel").mean() if len(novel_rows) else float("nan")

    per_seed_results.append({
        "seed": seed,
        "acc_3way": acc_3way,
        "acc_manual_builtin": acc_mb,
        "novel_recall": novel_recall,
        "n_test": len(results_s),
        "n_novel_test": len(novel_rows),
    })

summary_df = pd.DataFrame(per_seed_results)
print(summary_df)
print()
print(f"3-way accuracy:        mean={summary_df['acc_3way'].mean():.3f}  std={summary_df['acc_3way'].std():.3f}")
print(f"manual/builtin acc:    mean={summary_df['acc_manual_builtin'].mean():.3f}  std={summary_df['acc_manual_builtin'].std():.3f}")
print(f"novel recall:          mean={summary_df['novel_recall'].mean():.3f}  std={summary_df['novel_recall'].std():.3f}")
# === Aggregated confusion matrix / precision-recall across all seeds ===
pooled = pd.concat(all_predictions, ignore_index=True)
print(f"\npooled predictions across {len(SEEDS)} seeds: {len(pooled)} total")

from sklearn.metrics import confusion_matrix, classification_report

cm_pooled = confusion_matrix(pooled["structural_label"], pooled["semantic_predicted"], labels=labels_order)
print("\npooled confusion matrix [rows=structural label, cols=semantic prediction]")
print(pd.DataFrame(cm_pooled, index=labels_order, columns=labels_order))

print("\npooled classification report (all seeds combined):")
print(classification_report(pooled["structural_label"], pooled["semantic_predicted"], labels=labels_order))

# Same thing, restricted to manual/builtin only -- this is the number that
# actually goes in the paper
pooled_mb = pooled[pooled["structural_label"].isin(["manual", "builtin"])]
print("\npooled manual/builtin-only report:")
print(classification_report(pooled_mb["structural_label"], pooled_mb["semantic_predicted"], labels=["manual", "builtin"]))

## 12. Figures from the multi-seed robustness check

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(9, 5))

x = summary_df["seed"]
mb_acc = summary_df["acc_manual_builtin"]
three_acc = summary_df["acc_3way"]

ax.plot(x, mb_acc, marker="o", color="#7a4d99", linewidth=2, label="manual/builtin accuracy", zorder=3)
ax.plot(x, three_acc, marker="s", color="#5b7a9d", linewidth=2, linestyle="--", label="3-way accuracy", zorder=3)

mb_mean, mb_std = mb_acc.mean(), mb_acc.std()
ax.axhline(mb_mean, color="#7a4d99", alpha=0.4, linewidth=1)
ax.fill_between(x, mb_mean - mb_std, mb_mean + mb_std, color="#7a4d99", alpha=0.12,
                 label=f"manual/builtin mean ± SD ({mb_mean:.3f} ± {mb_std:.3f})")

ax.set_xlabel("Random split (seed)")
ax.set_ylabel("Accuracy")
ax.set_title("Taxonomic-agreement accuracy across 10 random 70/30 splits")
ax.set_xticks(x)
ax.legend(loc="lower right", fontsize=9)
ax.set_ylim(0.75, 0.95)
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/fig_multiseed_accuracy_stability.png", dpi=200, bbox_inches="tight")
plt.show()

Figure B — 2*2 confusion matrix, as a proper heatmap:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix

labels_mb = ["manual", "builtin"]
cm_mb = confusion_matrix(pooled_mb["structural_label"], pooled_mb["semantic_predicted"], labels=labels_mb)
cm_mb_norm = cm_mb.astype(float) / cm_mb.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(5, 4.5))
im = ax.imshow(cm_mb_norm, cmap="Purples", vmin=0, vmax=1)

ax.set_xticks(range(len(labels_mb)))
ax.set_yticks(range(len(labels_mb)))
ax.set_xticklabels(labels_mb)
ax.set_yticklabels(labels_mb)
ax.set_xlabel("Semantic prediction")
ax.set_ylabel("Structural label (ground truth)")
ax.set_title(f"Pooled confusion matrix -- manual/builtin only\n({len(SEEDS)} splits, {cm_mb.sum()} predictions)")

for i in range(len(labels_mb)):
    for j in range(len(labels_mb)):
        count = cm_mb[i, j]
        pct = cm_mb_norm[i, j]
        color = "white" if pct > 0.5 else "black"
        ax.text(j, i, f"{count}\n({pct:.0%})", ha="center", va="center", color=color, fontsize=12)

fig.colorbar(im, ax=ax, label="row-normalized (recall)")
plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/fig_pooled_confusion_matrix_manual_builtin.png", dpi=200, bbox_inches="tight")
plt.show()

Figure C — precision/recall per class, pooled (bar chart):

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import precision_score, recall_score

precisions_mb = [precision_score(pooled_mb["structural_label"], pooled_mb["semantic_predicted"], labels=[l], average=None, zero_division=0)[0] for l in labels_mb]
recalls_mb = [recall_score(pooled_mb["structural_label"], pooled_mb["semantic_predicted"], labels=[l], average=None, zero_division=0)[0] for l in labels_mb]

x = np.arange(len(labels_mb))
width = 0.35

fig, ax = plt.subplots(figsize=(6, 5))
bars1 = ax.bar(x - width/2, precisions_mb, width, label="Precision", color="#3f7a54")
bars2 = ax.bar(x + width/2, recalls_mb, width, label="Recall", color="#a5423f")

for bars in (bars1, bars2):
    for b in bars:
        ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.02, f"{b.get_height():.2f}",
                 ha="center", fontsize=11)

ax.set_xticks(x)
ax.set_xticklabels(labels_mb)
ax.set_ylim(0, 1.15)
ax.set_ylabel("Score")
ax.set_title(f"Pooled precision/recall -- manual/builtin only\n({len(SEEDS)} splits, {len(pooled_mb)} predictions)")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PROJECT_DIR}/fig_pooled_precision_recall_manual_builtin.png", dpi=200, bbox_inches="tight")
plt.show()